In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files `under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -q "uvicorn==0.30.6"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 2.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
mcp 1.27.0 requires uvicorn>=0.31.1; sys_platform != "emscripten", but you have uvicorn 0.30.6 which is incompatible.
google-adk 1.29.0 requires uvicorn<1.0.0,>=0.34.0, but you have uvicorn 0.30.6 which is incompatible.


In [3]:
pip install -U bitsandbytes>=0.46.1

Note: you may need to restart the kernel to use updated packages.


In [4]:
!pip install -q pyngrok

from pyngrok import ngrok

In [ ]:
# =============================================================================
# شغّل الملف ده على Kaggle Notebook (فعّل GPU T4 x2 من الإعدادات على اليمين)
# انسخ كل خلية (# --- CELL --- ) في خلية منفصلة في الـ Notebook
# =============================================================================

# --- CELL 1: تثبيت المكتبات ---
# !pip install -q transformers accelerate bitsandbytes fastapi uvicorn pyngrok nest_asyncio python-multipart

# --- CELL 2: تحميل الموديل ---
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"  # أو "Qwen/Qwen2.5-3B-Instruct" لو عايز أخف وأسرع

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print("Loading tokenizer & model... (هياخد كام دقيقة أول مرة)")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
print("Model loaded successfully!")


# --- CELL 3: تعريف الـ API ---
import nest_asyncio
nest_asyncio.apply()

from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn

app = FastAPI()


class GenerateRequest(BaseModel):
    prompt: str
    max_new_tokens: int = 800
    temperature: float = 0.3


@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_NAME}


@app.post("/generate")
def generate(req: GenerateRequest):
    messages = [{"role": "user", "content": req.prompt}]
    chat_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=req.max_new_tokens,
            temperature=req.temperature,
            do_sample=req.temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return {"response": generated_text}


# --- CELL 4: تشغيل السيرفر وفتح نفق ngrok ---
from pyngrok import ngrok

# حط الـ Auth Token بتاعك من https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = "Your token here"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(8000)
print("=" * 60)
print(f"Public URL (حط ده في .env بتاعك في LOCAL_LLM_URL):")
print(public_url)
print("=" * 60)

uvicorn.run(app, host="0.0.0.0", port=8000)

Loading tokenizer & model... (هياخد كام دقيقة أول مرة)


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded successfully!
Public URL (حط ده في .env بتاعك في LOCAL_LLM_URL):
NgrokTunnel: "https://gruffly-derived-reverse.ngrok-free.dev" -> "http://localhost:8000"


INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
